# Climate-related health research funding: extraction to outputs
## Main approach + methodological bands | 1990–2025
Run this notebook from top to bottom. The live workflow extracts data from Dimensions,
constructs the eleven retained definitions, generates a **new inspection sample**, applies
the **separate fixed historical exclusions**, and writes tables and figures locally.

The API key is read from `.env`. 

Use `MODE = "demo"` for a fabricated, offline rehearsal; demo results are not report results.

Install the environment once using `docs/RUN_NOTEBOOK.md`.


## 0. Locate the project and choose the run
`report_check` is a resumable extraction directory. Keep it for an interrupted run;
choose a **new name** for a genuinely fresh extraction. Do not run two kernels against the
same directory. Repeating a completed extraction reuses its verified files, not the live API.

In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "run_pipeline.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the extracted repository's notebooks folder.")
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use a Python 3.11–3.13 environment; see docs/RUN_NOTEBOOK.md.")
sys.path.insert(0, str(ROOT / "src"))

from lancet_funding.workflow import load_config, extract_stage, analyse_stage
from lancet_funding.io import environment, read_table

MODE = "live"  # change only to "demo" for the offline rehearsal
RUN_NAME = "report_check"  # use report_check_02 for a NEW live extraction
if MODE not in {"live", "demo"}:
    raise ValueError("MODE must be live or demo.")
CFG = load_config()
RUN = ROOT / "runs" / (RUN_NAME if MODE == "live" else "synthetic_demo")
print("Years:", CFG["start_year"], "to", CFG["end_year"])
print("Main approach: 14; band scenarios:", CFG["scenarios"])
print("Local output directory:", RUN)
print("Environment:", environment())

## 1. Load the local methodological inputs

In [ ]:
from lancet_funding.reference import prepare_local_inputs

CLIENT = None
if MODE == "demo":
    from lancet_funding.demo import prepare_demo
    CFG, CLIENT = prepare_demo(CFG, RUN)
    print("Using fabricated grants and fabricated reference inputs.")
else:
    reference_status = prepare_local_inputs(CFG)
    print({name: entry["rows"] for name, entry in reference_status.items()})

## 2. Supply the API key without saving it in this notebook
The public placeholder is `XXXXXXX` in `.env.example`. For a persistent local key, copy
that file to `.env` and replace the placeholder **in `.env` only**. Otherwise the following
cell prompts privately. An API key must have Dimensions Analytics/grants permissions.

In [ ]:
if MODE == "live":
    from dotenv import load_dotenv
    from getpass import getpass
    load_dotenv(ROOT / ".env", override=False)
    if os.environ.get("DIMENSIONS_API_KEY", "").strip().upper() in {"", "XXXXXXX"}:
        os.environ["DIMENSIONS_API_KEY"] = getpass("Dimensions API key (not saved in notebook): ").strip()
    if os.environ["DIMENSIONS_API_KEY"].upper() in {"", "XXXXXXX"}:
        raise ValueError("A real eligible key is required for a live run.")
    print("API credential is present; its value will not be displayed.")
else:
    print("No API credential needed for the offline demo.")

## 3. Check authentication and current API fields
This uses a few small queries. A successful check is not a successful full extraction
and is not evidence of numerical agreement with the historical report.

In [ ]:
if MODE == "live":
    from lancet_funding.cli import main
    status = main(["doctor", "--live"])
    if status != 0:
        raise RuntimeError("The API diagnostic failed; resolve it before continuing.")
else:
    print("Live diagnostic skipped for synthetic mode.")

## 4. Extract numerators and denominators, 1990–2025
Four server-side climate-plus-health base searches establish H, A, R and U membership.
Grant-ID set operations create the eleven numerators. Twelve health-only conjunctions
supply denominator aggregates; inclusion–exclusion produces unions and the main approach.

Requests are paced and cached. There are roughly two thousand distinct queries before
extra pages/date partitions; the live extraction can take well over an hour. Keep the
computer awake. Restarting this cell reuses completed queries. A failed query is never
interpreted as a zero. Raw CSV, JSONL and XLSX outputs remain under the local run directory.

In [ ]:
RAW = extract_stage(CFG, RUN, client=CLIENT)
print("Complete, hash-verified extraction:", RAW)
scenario_manifest = read_table(RAW / "scenario_manifest.csv")
display(scenario_manifest[["scenario_id", "classification_exclusions_applied", "search_profile"]])

## 5. Regenerate the Scenario 14 top-five inspection workbook
This is the generator's ranking operation: group by the year of `start_date`, select the
five largest `funding_usd` awards, and export. The fresh file is saved to
`runs/<name>/review/top5_values.xlsx`, **not** `private/top5_values.xlsx`.
Transfer `top5_values.xlsx` from `runs/<name>/review/` to `private/` after careful analytical cleaning.
Then you can move to the next cells.

In [ ]:
from lancet_funding.review import generate_review
review = generate_review(CFG, RUN)
print("Fresh inspection rows:", len(review["sample"]))
print("Historical fixed rows:", len(review["reconciliation"]))
print("Fresh file:", RUN / "review" / "top5_values.xlsx")
display(review["reconciliation"][["year", "found_in_current_s14", "in_current_top5", "current_minus_historical_usd"]].head())

## 6. Clean, aggregate and generate all analysis outputs
Funder and recipient conventions remain separate, following the selected source blocks.
The default band reproduces the **all-eleven** min/max calculation in the historical plot;
its ten-alternative-only counterpart is also exported. They are methodological ranges,
not confidence intervals. The main figure now spans 1990–2025. A separate 2010–2024 crop
is retained for direct comparison with the original figure.

This cell rebuilds `results/`, but does not alter the raw extraction or historical input
workbooks. A complete figure set includes country, regional and scenario-family panels
and may take several minutes after extraction. All workbooks and chart source tables
are written locally.

In [ ]:
result = analyse_stage(
    CFG,
    RUN,
    figures=True,
    data_label="synthetic" if MODE == "demo" else "Dimensions live extract",
)

print(result["status"])
print("Master analytical workbook:", RUN / "results" / "ALL_TABLES.xlsx")
print("Figure gallery:", RUN / "results" / "figures" / "index.html")